# Your own model: fine-tune Qwen3-4B for Ollama

Trains a LoRA add-on for **Qwen3-4B** on `train.jsonl` (made by `build_dataset.py`), then saves the result as one
`.gguf` file that `install_to_ollama.py` turns into an Ollama model (`pokemon-tcg`).

**On Google Colab (free):**
1. *Runtime → Change runtime type → T4 GPU*, then *Save*.
2. *Runtime → Run all*. When it asks, upload `train.jsonl`.
3. Training takes about 1–2 hours for the Pokémon TCG data alone, and roughly 3–5 hours with everything from
   `build_knowledge.py` too. At the end the `.gguf` file (about 2.5 GB) downloads, or is copied to your Google Drive
   if you set `SAVE_TO_DRIVE` below (more reliable for big files).
4. With `SAVE_TO_DRIVE = True`, training also saves a checkpoint to your Drive every few hundred steps. If Colab
   disconnects, just *Run all* again (and upload the same `train.jsonl`): it carries on from the last checkpoint.

**On your own PC:** run this notebook in Jupyter under WSL2 (Unsloth needs Linux) with `train.jsonl` next to it.
A 6 GB card (RTX 2060) works with the settings below; close anything else using the GPU (the GPU learner, games).

In [ ]:
%%capture
# Unsloth: fast LoRA training that fits small GPUs. (Already installed? This just checks.)
!pip install unsloth

In [ ]:
# ---- Settings --------------------------------------------------------------------------------------------------
BASE_MODEL = "unsloth/Qwen3-4B"   # the same model as Ollama's qwen3:4b
MAX_SEQ_LENGTH = 2048             # longest example in tokens (the dataset's are well under this)
EPOCHS = 1                        # passes over the data: 1-2; more risks forgetting general skills
MAX_STEPS = 0                     # 0 = train the full EPOCHS; e.g. 60 for a quick test run first
GENERAL_SHARE = 0.15              # share of general conversations mixed in, so it stays good at normal chat
OUTPUT_NAME = "pokemon-tcg"
SAVE_TO_DRIVE = False             # True: checkpoints and the .gguf go to Google Drive (survives disconnects)
CHECKPOINT_EVERY = 250            # steps between checkpoints (when SAVE_TO_DRIVE is on)

In [ ]:
import os
OUTPUT_DIR = "outputs"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = f"/content/drive/MyDrive/llm-checkpoints/{OUTPUT_NAME}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
if not os.path.exists("train.jsonl"):
    from google.colab import files
    print("Upload train.jsonl (from PokemonTCGSim/LlmTraining):")
    files.upload()
print("train.jsonl:", os.path.getsize("train.jsonl") // 1024, "KB")

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
# LoRA: the base model stays frozen; only this small add-on learns (which is what keeps it from forgetting much).
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

In [ ]:
import json, random
from datasets import Dataset, load_dataset
from unsloth.chat_templates import standardize_data_formats

pokemon = [json.loads(line)["messages"] for line in open("train.jsonl", encoding="utf-8")]

# General conversations (a public instruction dataset), so it keeps its everyday skills.
general_count = int(len(pokemon) * GENERAL_SHARE / (1 - GENERAL_SHARE))
general = []
if general_count > 0:
    g = standardize_data_formats(load_dataset("mlabonne/FineTome-100k", split="train"))
    g = g.shuffle(seed=3407).select(range(min(general_count, len(g))))
    general = [[{"role": m["role"], "content": m["content"]} for m in convo] for convo in g["conversations"]]

convos = pokemon + general
random.Random(3407).shuffle(convos)
texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]
dataset = Dataset.from_dict({"text": texts})
print(f"{len(pokemon):,} examples from train.jsonl + {len(general):,} general = {len(dataset):,}")
print(texts[0][:800])

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=EPOCHS, max_steps=MAX_STEPS if MAX_STEPS > 0 else -1,
        learning_rate=2e-4, logging_steps=20, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir=OUTPUT_DIR, report_to="none",
        save_strategy="steps" if SAVE_TO_DRIVE else "no", save_steps=CHECKPOINT_EVERY, save_total_limit=2))
# Learn only from the answers, not from repeating the questions.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
# Carry on from the last checkpoint if an earlier run was cut off (same train.jsonl, same settings).
import glob
resume = SAVE_TO_DRIVE and bool(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")))
if resume:
    print("Resuming from the last checkpoint in", OUTPUT_DIR)
stats = trainer.train(resume_from_checkpoint=resume or None)
print(stats)

In [ ]:
# Try it: a couple of questions before saving.
FastLanguageModel.for_inference(model)
system = pokemon[0][0]["content"]
for q in ["Can Rare Candy be used on the first turn of the game?",
          "If my Active Pokémon is Asleep, can I retreat it?",
          "What type is Gengar and what is it weak to?",
          "Who is Edward in Thomas & Friends?",
          "How do I use Transform.Rotate in Unity?",
          "How do I install PyTorch with CUDA on Windows?",
          "What is the capital of France?"]:
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": q}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=200, temperature=0.7, top_p=0.8)
    print("Q:", q)
    print("A:", tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip(), "\n")

In [ ]:
# Save: the trained add-on merged into the model, as one 4-bit .gguf file for Ollama (takes several minutes).
import glob, shutil
model.save_pretrained_gguf(OUTPUT_NAME, tokenizer, quantization_method="q4_k_m")
ggufs = sorted(glob.glob(f"{OUTPUT_NAME}*/*.gguf") + glob.glob(f"{OUTPUT_NAME}*.gguf") + glob.glob("*.gguf"))
gguf = next((p for p in ggufs if "q4_k_m" in p.lower()), ggufs[-1])
final = f"{OUTPUT_NAME}.Q4_K_M.gguf"
if os.path.abspath(gguf) != os.path.abspath(final):
    shutil.copy(gguf, final)
print("saved", final, os.path.getsize(final) // (1024 * 1024), "MB")

In [ ]:
try:
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copy(final, "/content/drive/MyDrive/" + final)
        print("Copied to Google Drive: MyDrive/" + final)
    else:
        from google.colab import files
        files.download(final)
except ImportError:
    print("Not on Colab: the file is here:", os.path.abspath(final))
print("Then on your PC: python install_to_ollama.py", final)